In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 08_train_models
# MAGIC Entrenar RandomForest y GradientBoosting + MLflow

# COMMAND ----------

import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SPLIT_PATH = "/Volumes/olist/olist_gold/model_split/"

print("🚀 Iniciando entrenamiento de modelos\n")

# COMMAND ----------

# Configurar MLflow
mlflow.set_experiment("/olist_customer_premium")
print("✅ Experimento MLflow configurado\n")

# COMMAND ----------

# Cargar train y val
print("📥 Cargando datos...\n")

train_df = spark.read.format("delta").load(f"{SPLIT_PATH}train/").toPandas()
val_df = spark.read.format("delta").load(f"{SPLIT_PATH}val/").toPandas()

print(f"✅ Train: {len(train_df):,} registros")
print(f"✅ Val:   {len(val_df):,} registros\n")

# COMMAND ----------

# Separar X y y
print("🎯 Separando features y target...\n")

X_train = train_df.drop(columns=["is_premium"])
y_train = train_df["is_premium"]

X_val = val_df.drop(columns=["is_premium"])
y_val = val_df["is_premium"]

print(f"✅ X_train: {X_train.shape}")
print(f"✅ X_val:   {X_val.shape}\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Modelo 1: Random Forest

# COMMAND ----------

print("="*60)
print("🌲 RANDOM FOREST")
print("="*60)

with mlflow.start_run(run_name="RandomForest") as run:
    
    # Entrenar
    print("\n📊 Entrenando...\n")
    rf = RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    rf.fit(X_train, y_train)
    print("✅ Modelo entrenado\n")
    
    # Predecir
    y_pred = rf.predict(X_val)
    
    # Métricas
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    print(f"📈 Métricas en validación:")
    print(f"  • Accuracy: {acc:.4f}")
    print(f"  • F1-Score: {f1:.4f}\n")
    
    print("📋 Classification Report:")
    print(classification_report(y_val, y_pred))
    
    print("\n📊 Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred))
    print()
    
    # Log en MLflow
    mlflow.log_param("model_type", "RandomForest")
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 10)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    
    # Guardar modelo
    mlflow.sklearn.log_model(rf, "model")
    
    rf_f1 = f1
    rf_run_id = run.info.run_id
    
    print(f"✅ Modelo guardado en MLflow (run_id: {rf_run_id})\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Modelo 2: Gradient Boosting

# COMMAND ----------

print("="*60)
print("🚀 GRADIENT BOOSTING")
print("="*60)

with mlflow.start_run(run_name="GradientBoosting") as run:
    
    # Entrenar
    print("\n📊 Entrenando...\n")
    gb = GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        random_state=42
    )
    gb.fit(X_train, y_train)
    print("✅ Modelo entrenado\n")
    
    # Predecir
    y_pred = gb.predict(X_val)
    
    # Métricas
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    
    print(f"📈 Métricas en validación:")
    print(f"  • Accuracy: {acc:.4f}")
    print(f"  • F1-Score: {f1:.4f}\n")
    
    print("📋 Classification Report:")
    print(classification_report(y_val, y_pred))
    
    print("\n📊 Confusion Matrix:")
    print(confusion_matrix(y_val, y_pred))
    print()
    
    # Log en MLflow
    mlflow.log_param("model_type", "GradientBoosting")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    
    # Guardar modelo
    mlflow.sklearn.log_model(gb, "model")
    
    gb_f1 = f1
    gb_run_id = run.info.run_id
    
    print(f"✅ Modelo guardado en MLflow (run_id: {gb_run_id})\n")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Comparación y Mejor Modelo

# COMMAND ----------

print("="*60)
print("🏆 COMPARACIÓN DE MODELOS")
print("="*60)

print(f"\nRandom Forest:")
print(f"  • F1-Score: {rf_f1:.4f}")
print(f"  • Run ID: {rf_run_id}")

print(f"\nGradient Boosting:")
print(f"  • F1-Score: {gb_f1:.4f}")
print(f"  • Run ID: {gb_run_id}")

# Seleccionar mejor
if rf_f1 > gb_f1:
    best_model_name = "Random Forest"
    best_f1 = rf_f1
    best_run_id = rf_run_id
else:
    best_model_name = "Gradient Boosting"
    best_f1 = gb_f1
    best_run_id = gb_run_id

print(f"\n{'='*60}")
print(f"🥇 MEJOR MODELO: {best_model_name}")
print(f"{'='*60}")
print(f"F1-Score: {best_f1:.4f}")
print(f"Run ID: {best_run_id}")
print(f"\n💡 Para usar el modelo:")
print(f"   model = mlflow.sklearn.load_model('runs:/{best_run_id}/model')")

# COMMAND ----------

# MAGIC %md
# MAGIC ---
# MAGIC **Modelos entrenados y guardados en MLflow**
# MAGIC 
# MAGIC Ver experimento completo en:
# MAGIC - Workspace → Machine Learning → Experiments → `/olist_customer_premium`